In [1]:
import pandas as pd
from rdkit import Chem

def classify_acrylate(smiles):
    if pd.isna(smiles):
        return "Invalid", "Empty SMILES"
    
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return "Invalid", "RDKit could not parse"

    # Define the SMARTS pattern for an alpha-beta unsaturated ester/acid
    # Structure: C=C-C(=O)O
    # [CX3] matches sp2 carbons
    acrylate_pattern = Chem.MolFromSmarts('[CX3]=[CX3][CX3](=[OX1])[OX2]')
    
    if not mol.HasSubstructMatch(acrylate_pattern):
        return "Not Acrylate", "No acrylic moiety found"

    matches = mol.GetSubstructMatches(acrylate_pattern)
    
    # Flags to track what functional groups are present in the molecule
    has_acrylate = False
    has_methacrylate = False
    has_alpha_sub = False
    has_beta_sub = False
    
    for match in matches:
        # The SMARTS matches atoms in order: 
        # Index 0: Beta Carbon (end of double bond)
        # Index 1: Alpha Carbon (middle of double bond)
        # Index 2: Carbonyl Carbon
        
        beta_idx = match[0]
        alpha_idx = match[1]
        carbonyl_idx = match[2]
        
        beta_atom = mol.GetAtomWithIdx(beta_idx)
        alpha_atom = mol.GetAtomWithIdx(alpha_idx)
        
        # Check Beta Hydrogens to determine if it is a terminal alkene
        # GetTotalNumHs() counts both implicit and explicit Hydrogens
        beta_h_count = beta_atom.GetTotalNumHs()
        
        if beta_h_count == 2:
            # It is a Terminal Alkene (CH2=...)
            alpha_h_count = alpha_atom.GetTotalNumHs()
            
            if alpha_h_count == 1:
                # Structure: CH2=CH-COO...
                has_acrylate = True
            else:
                # Structure: CH2=C(R)-COO... (Alpha Substituted)
                # Check if the substituent is a Methyl group (Methacrylate)
                is_methyl = False
                for neighbor in alpha_atom.GetNeighbors():
                    nid = neighbor.GetIdx()
                    # Ignore the beta carbon and the carbonyl carbon
                    if nid == beta_idx or nid == carbonyl_idx: 
                        continue
                    
                    # Check if neighbor is a Methyl group (Carbon with 3 Hydrogens)
                    if neighbor.GetAtomicNum() == 6 and neighbor.GetTotalNumHs() == 3:
                        is_methyl = True
                
                if is_methyl:
                    has_methacrylate = True
                else:
                    has_alpha_sub = True
                    
        else:
            # It is an Internal Alkene (R-CH=... or R2C=...)
            # This covers Cinnamates, Crotonates, etc.
            has_beta_sub = True

    # Priority Hierarchy for final classification
    if has_acrylate:
        return "Acrylate", "Contains terminal CH2=CH-COO"
    elif has_methacrylate:
        return "Methacrylate", "Contains terminal CH2=C(Me)-COO"
    elif has_alpha_sub:
        return "Alpha-Substituted Acrylate", "Contains terminal CH2=C(R)-COO"
    elif has_beta_sub:
        return "Beta-Substituted", "Internal double bond (e.g. Cinnamate)"
    else:
        return "Uncertain", "Matched pattern but logic fell through"

# --- Main Execution ---

# 1. Load Data
df = pd.read_csv('acrylates.csv')

# 2. Apply Classification
# We use apply to run the function on every row
results = df['smiles'].apply(classify_acrylate)

# 3. Unpack results into new columns
df['Category'] = [r[0] for r in results]
df['Reason'] = [r[1] for r in results]

# 4. Save results
df.to_csv('acrylates_classified.csv', index=False)
print("Classification complete. Saved to 'acrylates_classified.csv'.")
print(df['Category'].value_counts())

[14:47:48] WARNING: not removing hydrogen atom without neighbors


Classification complete. Saved to 'acrylates_classified.csv'.
Category
Beta-Substituted              7095
Acrylate                      1639
Alpha-Substituted Acrylate     614
Not Acrylate                   237
Methacrylate                    41
Name: count, dtype: int64


In [5]:
df[df['Category'] == 'Acrylate'].drop(columns=['Category', 'Reason']).to_csv('../polygraphpy/data/full_dataset.csv', index=False)